# Lecture 18 - Graphs and DAGs

**Lecture Topic**: The most general structure, the two ways to walk it, and the dependency graphs behind every data pipeline. The last lecture of Module C.

## Announcements

- Last Module C lecture. No class Wednesday (Veterans Day).
- Keep going on your project work.
- ClassBuzz poll for today: `XXXX`

## Recap: last time

- A **heap** is a complete binary tree in an array: instant min/max, `O(log n)` insert and remove.
- **Top-k** with a size-k heap; the classic **sorts** trade time against memory.
- **Stability** for multi-key sorts and reproducibility; **out-of-core merge** for data bigger than memory.

Today: **graphs**, the structure that contains all the others, and **DAGs**, which are what a data pipeline really is.

Today:
 - Graphs, and how to store them
 - Traversal: depth-first and breadth-first
 - Shortest paths
 - DAGs, topological sort, and dependency resolution
 - When "optimal" is really a heuristic

# Graphs

A **graph** is just **vertices** (nodes) connected by **edges**. That is general enough to model almost anything relational: road networks, social connections, citations, web links, and the dependencies between steps in a pipeline. A tree is simply a graph with no cycles and a root.

Edges can be **undirected** (a mutual friendship) or **directed** (A links to B, follows on social media, or must run before B).

## Two ways to store a graph

**Adjacency list**: each vertex keeps a list of its neighbours. Space is `O(|V| + |E|)`, which is small when edges are few.

**Adjacency matrix**: an `n`-by-`n` grid where entry `[i, j]` is 1 if there is an edge. Edge lookup is `O(1)`, but the space is always `O(n^2)`, whether or not the edges exist.

In [ ]:
import numpy as np
import pandas as pd

# adjacency list: vertex -> neighbours
graph = {
    "A": ["B", "C"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B"],
    "E": ["B", "F"],
    "F": ["C", "E"],
}

# the same graph as an adjacency matrix
nodes = sorted(graph)
pos = {n: i for i, n in enumerate(nodes)}
M = np.zeros((len(nodes), len(nodes)), dtype=int)
for u, nbrs in graph.items():
    for v in nbrs:
        M[pos[u], pos[v]] = 1

print(pd.DataFrame(M, index=nodes, columns=nodes))

### Question 1

Your graph has 1 million vertices but only a few edges per vertex (a sparse graph, like a road network). Which representation should you use?

 - A. Adjacency matrix, for O(1) edge lookups
 - B. Adjacency list, because the matrix would need O(n^2) space
 - C. Either one; they use the same memory
 - D. Neither; graphs cannot be that large

**B.** A million-vertex adjacency matrix is `10^12` entries, the same `n`-by-`n` memory blow-up from the complexity lecture, and it would be almost entirely zeros. The adjacency list stores only the edges that exist. Real large graphs use adjacency lists or sparse matrices (`scipy.sparse`) for exactly this reason.

## The same thing on real data: the taxi flow graph

Turn the taxi trips into a graph. Each zone is a vertex, and an edge from zone A to zone B carries the number of trips that went that way. Then compare storing it dense against storing it sparse.

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix

taxi = pd.read_csv("../data/2019_Yellow_Taxi_Trip_Data.csv")
zones = sorted(set(taxi["pulocationid"]) | set(taxi["dolocationid"]))
pos = {z: i for i, z in enumerate(zones)}
n = len(zones)

flow = taxi.groupby(["pulocationid", "dolocationid"]).size().reset_index(name="trips")
rows = flow["pulocationid"].map(pos).to_numpy()
cols = flow["dolocationid"].map(pos).to_numpy()
trips = flow["trips"].to_numpy()
sparse = coo_matrix((trips, (rows, cols)), shape=(n, n))

dense_bytes = n * n * 8                                   # a dense float matrix
sparse_bytes = sparse.data.nbytes + sparse.row.nbytes + sparse.col.nbytes
print(f"{n} zones, {sparse.nnz} edges that actually occur")
print(f"dense  n x n matrix: {n * n:,} cells, ~{dense_bytes / 1e3:.0f} KB")
print(f"sparse (edges only): ~{sparse_bytes / 1e3:.0f} KB")
print(f"dense / sparse: {dense_bytes / sparse_bytes:.0f}x")

The ratio is only a few times here, because 191 zones is small. But the dense cost grows as `n^2` while the sparse cost grows with the number of edges. Scale the same graph up to a million users and the dense matrix is `10^12` cells, about 8 TB, while the sparse one still holds only the edges that exist. This is why real graph tools (`scipy.sparse`, `networkx`) store the edges rather than an `n`-by-`n` grid.

# Traversal: depth-first and breadth-first

To explore a graph you visit vertices by following edges, taking care not to revisit. There are two orders, and you already know the structures behind them:

 - **Depth-first search (DFS)** goes as deep as it can before backtracking. It uses a **stack**.
 - **Breadth-first search (BFS)** explores in rings, all neighbours first, then their neighbours. It uses a **queue**.

Both visit every vertex and every edge once, so both are `O(|V| + |E|)`.

In [ ]:
from collections import deque

def bfs(graph, start):
    seen, order, q = {start}, [], deque([start])
    while q:
        u = q.popleft()                 # a queue: first in, first out
        order.append(u)
        for w in graph[u]:
            if w not in seen:
                seen.add(w)
                q.append(w)
    return order

def dfs(graph, start):
    seen, order, stack = set(), [], [start]
    while stack:
        u = stack.pop()                 # a stack: last in, first out
        if u in seen:
            continue
        seen.add(u)
        order.append(u)
        for w in reversed(graph[u]):     # reversed so we take neighbours in listed order
            if w not in seen:
                stack.append(w)
    return order

print("BFS from A:", bfs(graph, "A"))
print("DFS from A:", dfs(graph, "A"))

### Question 2

Depth-first search and breadth-first search are the same algorithm with one difference: the structure holding the vertices still to visit. Which is which?

 - A. DFS uses a queue; BFS uses a stack
 - B. DFS uses a stack; BFS uses a queue
 - C. Both use a stack
 - D. Both use a queue

**B. DFS uses a stack, BFS uses a queue.** Swapping the one structure flips the whole character of the search. A stack keeps diving down the most recent path; a queue fans out level by level. This is the stack and queue from the recursion lecture doing real work.

## Shortest paths

Because BFS explores in rings, the first time it reaches a vertex is by a path with the **fewest edges**. So BFS gives the shortest path in an unweighted graph for free: track how far each vertex is from the start.

In [ ]:
from collections import deque

def hops_from(graph, start):
    dist = {start: 0}
    q = deque([start])
    while q:
        u = q.popleft()
        for w in graph[u]:
            if w not in dist:
                dist[w] = dist[u] + 1
                q.append(w)
    return dist

print("edges from A to each vertex:", hops_from(graph, "A"))

When edges have **weights** (road distances, travel times), fewest-hops is not enough, and you use **Dijkstra's algorithm**: the same idea, but it pulls the closest unvisited vertex next using a **priority queue**, which is the heap from last time. The structures compound: a graph search powered by a heap.

We can run that on the taxi flow graph. Weight each edge by `1 / trips`, so heavily used routes count as shorter, and ask `scipy.sparse.csgraph` for the shortest weighted paths out of the busiest zone.

In [ ]:
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import dijkstra

weights = coo_matrix((1.0 / trips, (rows, cols)), shape=(n, n)).tocsr()
busiest = pos[taxi["pulocationid"].value_counts().index[0]]

dist = dijkstra(weights, directed=True, indices=busiest)
reachable = int(np.isfinite(dist).sum())
finite = dist[np.isfinite(dist) & (dist > 0)]
print(f"from the busiest zone, Dijkstra reaches {reachable} of {n} zones")
print(f"nearest reachable cost: {finite.min():.4f}   farthest: {finite.max():.4f}")

Dijkstra pulls the closest unvisited zone next, and "closest" is tracked with a priority queue, which is the heap from last lecture. In practice you reach for `scipy.sparse.csgraph` or `networkx`; the point for us is that a real graph search is a stack, a queue, or a heap doing the walking.

# DAGs: the shape of a pipeline

A **directed acyclic graph** (DAG) is a directed graph with no cycles. This is the single most important graph in data science, because a **dependency** is a directed edge: "clean" depends on "load", "features" depends on "clean", and so on. No cycles, because a step cannot depend on itself.

Your data pipeline is a DAG. So is a build system (`make`), a package manager resolving installs, a spreadsheet's cell references, and a scheduler like Airflow or Dask.

## Topological sort = a valid order to run things

A **topological sort** orders the vertices so that every edge points forward: every task comes after everything it depends on. That ordering *is* the execution plan, and finding it is `O(|V| + |E|)`. Python has it in the standard library.

In [ ]:
from graphlib import TopologicalSorter

# each task -> the tasks it depends on (must run first)
pipeline = {
    "clean":    ["load"],
    "features": ["clean"],
    "train":    ["features"],
    "report":   ["train", "features"],
}

order = list(TopologicalSorter(pipeline).static_order())
print("a valid run order:", order)

`load` comes first, `report` last, and every task lands after the things it needs. That is dependency resolution, the same job `pip` does working out install order and `make` does deciding what to rebuild.

### Question 3

What does a topological sort of a dependency DAG give you, and what does it mean if the sort **fails**?

 - A. It sorts tasks alphabetically; failure means a typo
 - B. A valid order to run the tasks; failure means there is a cycle (a circular dependency)
 - C. The fastest task first; failure means the graph is too big
 - D. A random order; it never fails

In [ ]:
from graphlib import TopologicalSorter, CycleError

circular = {"a": ["b"], "b": ["c"], "c": ["a"]}      # a -> b -> c -> a
try:
    list(TopologicalSorter(circular).static_order())
except CycleError as e:
    print("CycleError:", e)

**B.** A topological sort is a valid execution order, and it exists only if the graph is truly acyclic. A cycle is a circular dependency ("a needs b, b needs c, c needs a"), there is no order that satisfies it, and the sort reports it rather than looping forever. That error is how a build tool or a package manager tells you your dependencies are tangled.

# When "optimal" is really a heuristic

Not every graph question has a cheap answer. Some are **intractable**: no known algorithm solves them efficiently as they grow. The classic is the **travelling salesman problem**, the shortest route visiting every city once. Checking all routes is `O(n!)`, the fastest-growing cost from the complexity lecture.

In [ ]:
import math

for n in [5, 10, 20, 50]:
    routes = math.factorial(n - 1) // 2
    print(f"{n:2d} cities: {routes:.1e} possible routes")

At 50 cities there are more routes than atoms in your body, so no computer checks them all. Real tools return a **heuristic**: a route that is good, found quickly, but not provably the shortest. The same is true of optimal clustering, graph colouring, and many scheduling problems.

The judgment call here: when an agent or a library reports the "optimal" route, schedule, or clustering, ask whether the problem is actually tractable. Often "optimal" really means "a sensible heuristic", and knowing the difference is the point of being able to reason about the algorithm instead of trusting the label.

# Coda: orchestrating agents is a graph

All term we have treated an agent as a single loop. People increasingly wire several agents together, and once you do, "how are they connected?" is a graph question. The common setups have names you now recognise.

## The patterns are shapes you have seen

Anthropic's "Building Effective Agents" lists five ways to combine model calls. Each one is a graph from today:

| Pattern | What it does | The shape |
|---|---|---|
| **Prompt chaining** | one agent's output feeds the next | a path (a linear DAG) |
| **Routing** | a classifier sends the work down one branch | a tree |
| **Parallelisation** | split the work, run it at once, combine the results | fan-out and fan-in (a DAG) |
| **Orchestrator-workers** | a lead agent hands pieces to workers, then merges | a tree (the sub-agents from Lecture 12) |
| **Evaluator-optimizer** | one agent drafts, another critiques, repeat | a cycle, with a stopping rule |

The run order of a pipeline of agents is a topological sort, the same one we used for tasks.

### Question 4

Four of the five patterns have a fixed run order you could read off in advance. Which one does not, because it loops until some condition is met?

 - A. Prompt chaining
 - B. Routing
 - C. Parallelisation
 - D. Evaluator-optimizer

**D, evaluator-optimizer.** It is the one with a cycle (draft, critique, revise, back to critique), so there is no fixed order and no natural end. Like any cycle, you have to give it a stopping rule, for example a cap on the number of rounds or a quality threshold.

## The same topological sort, on agents

In [ ]:
from graphlib import TopologicalSorter

# a small multi-agent pipeline: each agent -> the agents it needs to run first
agents = {
    "draft":      ["research"],
    "fact_check": ["draft"],
    "critique":   ["draft"],
    "revise":     ["critique", "fact_check"],
    "finalize":   ["revise"],
}

print("run order:", list(TopologicalSorter(agents).static_order()))

`critique` and `fact_check` both depend only on `draft`, so they can run at the same time, which is the fan-out. The thing scheduling a multi-agent pipeline is doing exactly what `make` or Dask does with a task graph.

## Which shapes to trust

 - The orchestrations you can reason about are the **DAGs**: a clear direction, no loops, a run order you can read off, and a cost you can add up. The free-for-all where every agent talks to every other is the one you cannot debug.
 - The **cycles** (critique loops, debates) need a stopping rule, the same lesson as a cycle in a dependency graph: there is no natural end, so you set one.
 - Cost adds up. Each agent carries its own growing context from Lecture 13, so five workers are five of those, plus the coordination between them. More agents means more places to go wrong.
 - The standing advice is to start with one agent and add a second only when the task really needs it.

Choosing a pipeline of specialised agents or one agent that does everything is the same architecture choice we start on in Module D.

# Recap, and the end of Module C

 - A **graph** is vertices and edges; store it sparse (an adjacency list or `scipy.sparse`, holding only the real edges) rather than a dense `n`-by-`n` matrix.
 - **DFS** uses a stack, **BFS** uses a queue; both are `O(|V| + |E|)`, and BFS gives shortest unweighted paths, while **Dijkstra** (via a heap) handles weights, shown here on the taxi flow graph.
 - A **DAG** is a dependency graph; **topological sort** is the run order, and a failure means a **cycle**.
 - Some problems are **intractable**; a reported "optimal" is often a **heuristic**, and recognising that is the skill.
 - **Several agents wired together** form a graph too: pipelines and fan-out are DAGs you can schedule and cost, and critique loops are cycles that need a stopping rule.

That closes Module C: from counting operations, through arrays, hashes, trees and heaps, to graphs. You can now predict how code scales and question the structures an agent reaches for.

**Next (Module D, S21): software engineering** - architecture, structuring code, and testing what you did not write.